# Pipeline d'Analyse Vidéo de Football
Optimisation du suivi de joueurs et projection tactique 2D.

In [ ]:
# Installation des dépendances et imports bibliothèques
!pip -q install yt-dlp ultralytics supervision opencv-python matplotlib seaborn pandas numpy scikit-learn filterpy deep-sort-realtime "lap>=0.5.12"

import json
import math
import time
import warnings
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from sklearn.cluster import KMeans
from filterpy.kalman import KalmanFilter

warnings.filterwarnings("ignore")

In [ ]:
# Variables globales et paramètres de configuration
VIDEO_URL = "https://youtu.be/78J5KzLCeOQ"
RAW_VIDEO = "match_full.mp4.mkv"
VIDEO_PATH = "video_test.mp4"
START_TIME = "00:18:00"
DURATION = "00:01:00"

# Dimensions du terrain (normes FIFA)
PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

# Résolution de référence pour le calcul des échelles
REFERENCE_WIDTH = 1920
REFERENCE_HEIGHT = 1080

# Paramètres d'analyse et de validation physique
CALIBRATION_KEYFRAMES = [0, 150, 300, 450, 600, 750, 900, 1050, 1200, 1350, 1500, 1650]
USE_AUTOMATIC_HOMOGRAPHY = False
USE_MANUAL_FALLBACK = True
DEEPSORT_MAX_FRAMES = 300

MAX_SPEED_M_S = 10.0
MAX_ACCEL_M_S2 = 6.0
MAX_TRACK_GAP_FRAMES = 5

OUTPUT_DIR = Path("resultats")

# Configuration de l'affichage
TEAM_DISPLAY_LABELS = {0: "Équipe A", 1: "Équipe B", -1: "Non classé / autre"}
TEAM_COLORS_HEX = {0: "#1e88e5", 1: "#e53935", -1: "#fdd835"}

In [ ]:
# Coordonnees theoriques des points du terrain aux normes FIFA
FIELD_POINTS = {
    "corner_top_left": (0.0, 0.0),
    "corner_top_right": (105.0, 0.0),
    "corner_bottom_right": (105.0, 68.0),
    "corner_bottom_left": (0.0, 68.0),
    "halfway_top": (52.5, 0.0),
    "halfway_bottom": (52.5, 68.0),
    "center_spot": (52.5, 34.0),
    "left_penalty_top_left": (0.0, 13.84),
    "left_penalty_top_right": (16.5, 13.84),
    "left_penalty_bottom_right": (16.5, 54.16),
    "left_penalty_bottom_left": (0.0, 54.16),
    "right_penalty_top_left": (88.5, 13.84),
    "right_penalty_top_right": (105.0, 13.84),
    "right_penalty_bottom_right": (105.0, 54.16),
    "right_penalty_bottom_left": (88.5, 54.16),
    "center_circle_top": (52.5, 24.85),
    "center_circle_bottom": (52.5, 43.15),
    "center_circle_left": (43.35, 34.0),
    "center_circle_right": (61.65, 34.0),
}

## 1. Acquisition et Préparation Vidéo
Extraction de la séquence cible et calcul des ratios de résolution.

In [ ]:
# Recuperation de la source video via yt-dlp
import os

if not os.path.exists(VIDEO_PATH):
    print("Telechargement de la video...")
    cookies_opt = "--cookies cookies.txt" if os.path.exists("cookies.txt") else ""
    !yt-dlp {cookies_opt} --no-check-certificates --user-agent "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36" --referer "https://www.google.com/" -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]" "{VIDEO_URL}" -o "{RAW_VIDEO}"

    # Traitement ffmpeg si le fichier source existe
    if os.path.exists(RAW_VIDEO):
        !ffmpeg -y -i "{RAW_VIDEO}" -ss {START_TIME} -t {DURATION} -c:v libx264 -c:a aac "{VIDEO_PATH}"
else:
    print(f"La video '{VIDEO_PATH}' existe deja. Etape de telechargement et decoupe ignoree.")

# Lecture des proprietes de la video
cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS) or 25
FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video: {FRAME_COUNT} frames, {VIDEO_FPS:.2f} FPS, resolution {WIDTH}x{HEIGHT}")

if FRAME_COUNT <= 0 or WIDTH <= 0 or HEIGHT <= 0:
    raise ValueError("Echec de la recuperation video.")

# Creation des repertoires de sortie
OUTPUT_DIR.mkdir(exist_ok=True)
viz_dir = OUTPUT_DIR / "visualisations"
viz_dir.mkdir(exist_ok=True)

# Calcul du ratio de mise a l'echelle
SCALE_X = WIDTH / REFERENCE_WIDTH
SCALE_Y = HEIGHT / REFERENCE_HEIGHT

def scale_image_points(points_list):
    return [(name, x_px * SCALE_X, y_px * SCALE_Y) for name, x_px, y_px in points_list]


## 2. Calibration Géométrique (Vision par Ordinateur)
Détection automatique des lignes de terrain et identification des intersections.

In [ ]:
# Fonctions de detection et classification des lignes par vision par ordinateur
def detect_pitch_lines(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_white = np.array([0, 0, 200])
    upper_white = np.array([180, 50, 255])
    mask_white = cv2.inRange(hsv, lower_white, upper_white)
    kernel = np.ones((3, 3), np.uint8)
    mask_white = cv2.morphologyEx(mask_white, cv2.MORPH_CLOSE, kernel)
    edges = cv2.Canny(mask_white, 30, 100)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=120, minLineLength=80, maxLineGap=30)
    return mask_white, edges, lines

def classify_lines(lines, img_shape):
    h, w = img_shape[:2]
    horizontales, verticales = [], []
    if lines is None:
        return horizontales, verticales
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = abs(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
        if angle < 30 or angle > 150:
            horizontales.append((x1, y1, x2, y2))
        elif 60 < angle < 120:
            verticales.append((x1, y1, x2, y2))
    return horizontales, verticales

def line_intersection(l1, l2):
    x1, y1, x2, y2 = l1
    x3, y3, x4, y4 = l2
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(denom) < 1e-6:
        return None
    px = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denom
    py = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denom
    return (int(px), int(py))

def find_intersections(horiz, vert, img_shape):
    h, w = img_shape[:2]
    pts = []
    for hline in horiz:
        for vline in vert:
            pt = line_intersection(hline, vline)
            if pt is not None and 0 < pt[0] < w and 0 < pt[1] < h:
                pts.append(pt)
    return pts

def auto_calibrate_frame(frame, frame_idx):
    mask, edges, lines = detect_pitch_lines(frame)
    horiz, vert = classify_lines(lines, frame.shape)
    intersections = find_intersections(horiz, vert, frame.shape)
    return {
        "frame_idx": frame_idx,
        "lines": lines,
        "horiz": horiz,
        "vert": vert,
        "intersections": intersections,
        "mask": mask,
        "edges": edges,
    }

In [ ]:
# Initialisation du dossier de keyframes et extraction des images pour calibration
frames_dir = Path("calibration_keyframes")
frames_dir.mkdir(exist_ok=True)

valid_keyframes = [f for f in CALIBRATION_KEYFRAMES if f < FRAME_COUNT]
calib_data = {}

for frame_idx in valid_keyframes:
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        continue

    cv2.imwrite(str(frames_dir / f"frame_{frame_idx:06d}.jpg"), frame)
    calib_data[frame_idx] = auto_calibrate_frame(frame, frame_idx)

print(f"Keyframes analysees: {len(calib_data)}/{len(valid_keyframes)}")

## 3. Matrice d'Homographie
Calcul du passage des coordonnées pixels aux coordonnées réelles (mètres).

In [ ]:
# Calcul de l'homographie via points automatiques ou manuels de secours
def match_auto_points(calib, field_points_ordered):
    if not USE_AUTOMATIC_HOMOGRAPHY:
        return None
    pts_img = calib["intersections"]
    if len(pts_img) < 4:
        return None
    pts_img = pts_img[:4]
    pts_terrain = list(field_points_ordered.values())[:4]
    pts_img_np = np.array(pts_img, dtype=np.float32)
    pts_terrain_np = np.array(pts_terrain, dtype=np.float32)
    H, mask = cv2.findHomography(pts_img_np, pts_terrain_np, method=cv2.RANSAC, ransacReprojThreshold=5.0)
    return H

FIELD_POINTS_ORDERED = {
    "corner_top_left": (0.0, 0.0),
    "corner_top_right": (105.0, 0.0),
    "corner_bottom_right": (105.0, 68.0),
    "corner_bottom_left": (0.0, 68.0),
}

IMAGE_POINTS_BY_FRAME_FALLBACK = {
    150: [
        ("halfway_top", 1804, 399),
        ("corner_top_left", 509, 346),
        ("left_penalty_top_left", 310, 377),
        ("center_circle_left", 1305, 557),
        ("center_circle_bottom", 1783, 722),
    ],
    300: [
        ("halfway_top", 1502, 478),
        ("left_penalty_top_right", 108, 495),
        ("center_circle_left", 874, 718),
        ("center_circle_bottom", 1495, 893),
    ],
    600: [
        ("halfway_top", 1426, 209),
        ("corner_top_left", 495, 180),
        ("center_circle_bottom", 1408, 438),
        ("center_circle_right", 1794, 374),
    ],
    750: [
        ("halfway_bottom", 1154, 1076),
        ("corner_top_left", 337, 146),
        ("halfway_top", 1207, 159),
        ("center_circle_right", 1523, 300),
        ("left_penalty_top_left", 193, 172),
    ],
    1200: [
        ("center_circle_left", 10, 710),
        ("halfway_top", 627, 446),
        ("center_circle_right", 1218, 662),
        ("center_circle_bottom", 637, 835),
    ],
}

def compute_homography_for_frame(points_for_frame):
    src_px, dst_m = [], []
    for name, x_px, y_px in points_for_frame:
        if name not in FIELD_POINTS: continue
        src_px.append([float(x_px), float(y_px)])
        dst_m.append(FIELD_POINTS[name])
    src_px = np.array(src_px, dtype=np.float32)
    dst_m = np.array(dst_m, dtype=np.float32)
    if len(src_px) < 4: raise ValueError(f"4 points minimum requis")
    H, _ = cv2.findHomography(src_px, dst_m, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    return H

# Interpolation temporelle des matrices d'homographie entre les keyframes
H_BY_FRAME = {}
H_SOURCE_BY_FRAME = {}
for frame_idx in valid_keyframes:
    H = None
    if frame_idx in calib_data:
        H = match_auto_points(calib_data[frame_idx], FIELD_POINTS_ORDERED)
    if H is None and USE_MANUAL_FALLBACK and frame_idx in IMAGE_POINTS_BY_FRAME_FALLBACK:
        scaled_points = scale_image_points(IMAGE_POINTS_BY_FRAME_FALLBACK[frame_idx])
        H = compute_homography_for_frame(scaled_points)
    if H is not None:
        H_BY_FRAME[frame_idx] = H

## 4. Visualisation de la Calibration
Validation visuelle de la détection des points de repère sur les images clés.

In [ ]:
n_visu = min(len(valid_keyframes), 6)
fig, axes = plt.subplots(n_visu, 2, figsize=(20, 4 * n_visu))
if n_visu == 1: axes = axes.reshape(1, -1)

for idx, frame_idx in enumerate(valid_keyframes[:n_visu]):
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok: continue

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    axes[idx, 0].imshow(frame_rgb)
    axes[idx, 0].set_title(f"Frame {frame_idx} - Originale")
    axes[idx, 0].axis("off")

    cal = calib_data.get(frame_idx)
    if cal:
        img_lines = frame_rgb.copy()
        for pt in cal["intersections"]:
            cv2.circle(img_lines, pt, 6, (255, 0, 0), -1)
        if cal["lines"] is not None:
            for line in cal["lines"]:
                x1, y1, x2, y2 = line[0]
                cv2.line(img_lines, (x1, y1), (x2, y2), (0, 255, 0), 2)
        axes[idx, 1].imshow(img_lines)
        axes[idx, 1].set_title(f"Frame {frame_idx} - {len(cal['intersections'])} intersections")
    axes[idx, 1].axis("off")

plt.tight_layout()
plt.savefig(str(viz_dir / "keyframes_calibration.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(viz_dir / "keyframes_calibration.svg"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Détection et Suivi Multi-Objets (MOT)
Implémentation de YOLOv8 combiné à ByteTrack pour le tracking des joueurs.

In [ ]:
# Traitement de la video par YOLOv8 et ByteTrack pour le suivi des joueurs
model = YOLO("yolov8m.pt")
rows = []
start = time.time()

for frame_idx, r in enumerate(model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", persist=True, stream=True, save=False, classes=[0], verbose=False)):
    if r.boxes is None or r.boxes.id is None: continue
    boxes = r.boxes.xyxy.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    for box, track_id, conf in zip(boxes, ids, confs):
        x_min, y_min, x_max, y_max = box
        x_px = float((x_min + x_max) / 2)
        y_px = float(y_max)
        rows.append({
            "frame": frame_idx,
            "time_s": frame_idx / VIDEO_FPS,
            "tracker_id": int(track_id),
            "confidence": float(conf),
            "x_min": float(x_min),
            "y_min": float(y_min),
            "x_max": float(x_max),
            "y_max": float(y_max),
            "x_px": x_px,
            "y_px": y_px,
        })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(str(OUTPUT_DIR / "resultats_bruts_tracking.csv"), index=False)

# Affichage des premieres lignes du tableau brut de tracking
print(f"Detections trackees: {len(df_raw)} lignes")
df_raw.head()


In [ ]:
# Annotation video avec numeros de tracking
# Genere une video avec les boites de detection et les IDs de tracking
print("Generation de la video annotee avec tracking...")

# Preparer les donnees par frame pour acces rapide
frame_data = df_raw.groupby("frame").apply(
    lambda x: list(zip(x["x_min"], x["y_min"], x["x_max"], x["y_max"], x["tracker_id"]))
).to_dict()

# Lire la video source
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Definir le codec et creer le writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_path = str(OUTPUT_DIR / "video_annotee_tracking.mp4")
video_writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

# Couleurs pour les differents IDs (couleur aleatoire par ID)
np.random.seed(42)
colors = {}

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Recuperer les detections pour cette frame
    if frame_idx in frame_data:
        for x_min, y_min, x_max, y_max, track_id in frame_data[frame_idx]:
            x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)

            # Generer une couleur coherente par ID
            if track_id not in colors:
                colors[track_id] = tuple(np.random.randint(0, 255, 3).tolist())

            color = colors[track_id]

            # Dessiner la boite
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)

            # Ajouter le numero de tracking au-dessus de la boite
            cv2.putText(
                frame,
                f"ID: {int(track_id)}",
                (x_min, y_min - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    # Ecrire la frame dans la video de sortie
    video_writer.write(frame)
    frame_idx += 1

cap.release()
video_writer.release()

print(f"Video annotee sauvegardee: {out_path}")
print(f"Frames traitees: {frame_idx}")
print(f"Nombre d'IDs uniques: {len(colors)}")


## 6. Transformation Spatiale et Cinématique
Projection des trajectoires sur le plan du terrain et calcul des vitesses.

In [ ]:
# Projection des coordonnees pixels en coordonnees terrain (metres)
def pixel_to_meter(x_px, y_px, frame_idx, h_by_frame):
    keys = sorted(h_by_frame.keys())

    point = np.array(
        [[[float(x_px), float(y_px)]]],
        dtype=np.float32
    )

    if frame_idx <= keys[0]:
        projected = cv2.perspectiveTransform(
            point,
            h_by_frame[keys[0]]
        )[0, 0]

        return float(projected[0]), float(projected[1])

    if frame_idx >= keys[-1]:
        projected = cv2.perspectiveTransform(
            point,
            h_by_frame[keys[-1]]
        )[0, 0]

        return float(projected[0]), float(projected[1])

    for left, right in zip(keys[:-1], keys[1:]):
        if left <= frame_idx <= right:
            alpha = (
                (frame_idx - left)
                / (right - left)
            )

            left_position = cv2.perspectiveTransform(
                point,
                h_by_frame[left]
            )[0, 0]

            right_position = cv2.perspectiveTransform(
                point,
                h_by_frame[right]
            )[0, 0]

            projected = (
                (1 - alpha) * left_position
                + alpha * right_position
            )

            return float(projected[0]), float(projected[1])

    raise ValueError(
        f"Aucune homographie disponible pour la frame {frame_idx}"
    )

# Calcul des distances, vitesses et deltas temporels
df = df_raw.copy()
coords = df.apply(lambda row: pixel_to_meter(row["x_px"], row["y_px"], row["frame"], H_BY_FRAME), axis=1, result_type="expand")
df["x_m"], df["y_m"] = coords[0], coords[1]

df = df.sort_values(["tracker_id", "frame"])
df["dt_s"] = df.groupby("tracker_id")["time_s"].diff()
df["dist_m"] = np.sqrt(df.groupby("tracker_id")["x_m"].diff()**2 + df.groupby("tracker_id")["y_m"].diff()**2)
df["speed_km_h"] = (df["dist_m"] / df["dt_s"]) * 3.6

# Affichage des premieres lignes du tableau projete
df.head()


## 7. Lissage des Trajectoires (Filtre de Kalman)
Réduction du bruit de mesure et correction des anomalies de vitesse/accélération.

In [ ]:
def init_kalman(first_x, first_y):
    kf = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0 / VIDEO_FPS

    kf.F = np.array([
        [1, 0, dt, 0],
        [0, 1, 0, dt],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])

    kf.H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ])

    kf.x = np.array([
        [first_x],
        [first_y],
        [0.0],
        [0.0],
    ])

    kf.P *= 100.0
    kf.R = np.eye(2) * 5.0
    kf.Q = np.eye(4) * 0.1

    return kf

def apply_kalman_smoothing(track_df):
    if len(track_df) < 3:
        return track_df

    track_df = track_df.sort_values("frame").copy()

    first_x = float(track_df.iloc[0]["x_m"])
    first_y = float(track_df.iloc[0]["y_m"])
    kf = init_kalman(first_x, first_y)

    xs, ys = [], []

    for _, row in track_df.iterrows():
        kf.predict()
        kf.update(np.array([[row["x_m"]], [row["y_m"]]]))
        xs.append(float(kf.x[0, 0]))
        ys.append(float(kf.x[1, 0]))

    track_df["x_m"] = xs
    track_df["y_m"] = ys

    return track_df

df_kalman_list = []
for tid, group in df.groupby("tracker_id"):
    df_kalman_list.append(apply_kalman_smoothing(group))
df = pd.concat(df_kalman_list, ignore_index=True)

df = df.sort_values(["tracker_id", "frame"]).copy()
df["prev_x_m"] = df.groupby("tracker_id")["x_m"].shift(1)
df["prev_y_m"] = df.groupby("tracker_id")["y_m"].shift(1)
df["prev_time_s"] = df.groupby("tracker_id")["time_s"].shift(1)
df["dt_s"] = df["time_s"] - df["prev_time_s"]
df["frame_gap"] = df.groupby("tracker_id")["frame"].diff()
df["dist_m"] = np.sqrt((df["x_m"] - df["prev_x_m"])**2 + (df["y_m"] - df["prev_y_m"])**2)
df.loc[df["dt_s"].isna() | (df["dt_s"] <= 0), "dist_m"] = 0
df.loc[df["frame_gap"] > MAX_TRACK_GAP_FRAMES, "dist_m"] = np.nan
df["speed_m_s"] = df["dist_m"] / df["dt_s"]
df["speed_km_h"] = df["speed_m_s"] * 3.6

df["position_dans_terrain"] = (
    df["x_m"].between(0, PITCH_LENGTH_M)
    & df["y_m"].between(0, PITCH_WIDTH_M)
)

df["position_precedente_dans_terrain"] = (
    df.groupby("tracker_id")["position_dans_terrain"]
    .shift(1)
    .fillna(False)
)

df["segment_dans_terrain"] = (
    df["position_dans_terrain"]
    & df["position_precedente_dans_terrain"]
)

df.loc[
    ~df["segment_dans_terrain"],
    ["dist_m", "speed_m_s", "speed_km_h"]
] = np.nan

df["x_m_valide"] = df["x_m"].where(
    df["position_dans_terrain"]
)

df["y_m_valide"] = df["y_m"].where(
    df["position_dans_terrain"]
)

df["dist_m_raw"] = df["dist_m"]
df["speed_m_s_raw"] = df["speed_m_s"]
df["speed_km_h_raw"] = df["speed_km_h"]

df.loc[df["speed_m_s"] > MAX_SPEED_M_S, ["dist_m", "speed_m_s", "speed_km_h"]] = np.nan
df["prev_speed_m_s"] = df.groupby("tracker_id")["speed_m_s"].shift(1)
df["accel_m_s2"] = (df["speed_m_s"] - df["prev_speed_m_s"]) / df["dt_s"]
df["accel_m_s2_raw"] = df["accel_m_s2"]
df.loc[df["accel_m_s2"].abs() > MAX_ACCEL_M_S2, ["dist_m", "speed_m_s", "speed_km_h"]] = np.nan

df.to_csv(str(OUTPUT_DIR / "resultats_metres_kalman.csv"), index=False)

# Affichage des premieres lignes apres lissage de Kalman
df.head()


## 8. Classification des Équipes (Clustering)
Identification automatique des équipes par analyse de la couleur dominante des maillots (HSV).

In [ ]:
# Extraction de la couleur dominante via ROI joueur avec filtrage de la pelouse
def extract_team_color(frame, x_min, y_min, x_max, y_max):
    x1, y1 = max(0, int(x_min)), max(0, int(y_min))
    x2, y2 = min(frame.shape[1] - 1, int(x_max)), min(frame.shape[0] - 1, int(y_max))
    if x2 <= x1 or y2 <= y1: return None
    bw, bh = x2 - x1, y2 - y1

    # On cible uniquement le torse (du 15% au 55% de la hauteur) pour eviter le short/chaussettes/terrain
    tx1, tx2 = int(x1 + 0.25 * bw), int(x1 + 0.75 * bw)
    ty1, ty2 = int(y1 + 0.15 * bh), int(y1 + 0.55 * bh)
    roi = frame[ty1:ty2, tx1:tx2]
    if roi.size == 0: return None

    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV).reshape(-1, 3)

    # Masque pour exclure les teintes vertes de la pelouse (Teinte entre 35 et 85, Saturation > 30)
    is_grass = (hsv_roi[:, 0] >= 35) & (hsv_roi[:, 0] <= 85) & (hsv_roi[:, 1] > 30)
    jersey_pixels = hsv_roi[~is_grass]

    if len(jersey_pixels) < 10:
        jersey_pixels = hsv_roi # Repli si trop peu de pixels

    return np.median(jersey_pixels, axis=0)

# Clustering des equipes base sur un echantillon de frames
def assign_teams_ultra_fast(df_tracks, video_path, n_clusters=2, max_frames_to_read=None):
    if max_frames_to_read is None:
        max_frames_to_read = FRAME_COUNT
    sample_df = df_tracks[df_tracks['frame'] < max_frames_to_read].groupby('tracker_id').head(3)
    unique_frames = set(sample_df['frame'].unique())

    team_colors_samples = {}
    cap = cv2.VideoCapture(video_path)

    curr_frame = 0
    while curr_frame < max_frames_to_read:
        ok, frame = cap.read()
        if not ok: break

        if curr_frame in unique_frames:
            players_in_frame = sample_df[sample_df['frame'] == curr_frame]
            for _, row in players_in_frame.iterrows():
                tid = row['tracker_id']
                clr = extract_team_color(frame, row['x_min'], row['y_min'], row['x_max'], row['y_max'])
                if clr is not None:
                    if tid not in team_colors_samples: team_colors_samples[tid] = []
                    team_colors_samples[tid].append(clr)
        curr_frame += 1
    cap.release()

    final_colors = {tid: np.mean(clrs, axis=0) for tid, clrs in team_colors_samples.items()}
    if len(final_colors) < n_clusters: return {tid: -1 for tid in df_tracks['tracker_id'].unique()}

    color_matrix = np.array(list(final_colors.values()))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(color_matrix)

    # Garantie de cohérence de l'affectation des équipes.
    # Le cluster présentant la saturation moyenne la plus faible devient l'Équipe A,
    # tandis que le cluster le plus saturé devient l'Équipe B.
    cluster_centers = kmeans.cluster_centers_
    saturations = [center[1] for center in cluster_centers]
    if saturations[0] > saturations[1]:
        labels = [1 - l for l in labels]

    return {tid: int(label) for tid, label in zip(final_colors.keys(), labels)}

# Application du clustering
team_map = assign_teams_ultra_fast(df, VIDEO_PATH, n_clusters=2, max_frames_to_read=FRAME_COUNT)
df["team"] = df["tracker_id"].map(team_map).fillna(-1).astype(int)


## 9. Analyse Statistique
Calcul des indicateurs de performance (KPI) individuels et collectifs.

In [ ]:
stats_joueurs = df.groupby(["tracker_id", "team"]).agg(
    frames_observees=("frame", "count"),
    duree_s=("time_s", lambda s: float(s.max() - s.min()) if len(s) else 0),
    distance_totale_m=("dist_m", "sum"),
    vitesse_max_km_h=("speed_km_h", "max"),
    vitesse_moyenne_km_h=("speed_km_h", "mean"),
    confiance_moyenne=("confidence", "mean"),
    x_moyen_m=("x_m_valide", "mean"),
    y_moyen_m=("y_m_valide", "mean"),
).round(2)

stats_joueurs = stats_joueurs.sort_values("frames_observees", ascending=False)
stats_joueurs.to_csv(str(OUTPUT_DIR / "stats_joueurs.csv"))

stats_equipe = df.groupby("team").agg(
    identifiants_uniques=("tracker_id", "nunique"),
    distance_totale_equipe_m=("dist_m", "sum"),
    vitesse_max_equipe_km_h=("speed_km_h", "max"),
    vitesse_moy_equipe_km_h=("speed_km_h", "mean"),
).round(2)
stats_equipe.to_csv(str(OUTPUT_DIR / "stats_equipes.csv"))

# Affichage des statistiques generales generees
print("=== STATISTIQUES INDIVIDUELLES (Top 10) ===")
display(stats_joueurs.head(10))
print("\n=== STATISTIQUES PAR EQUIPE ===")
display(stats_equipe)


## 10. Validation de la Précision
Calcul de l'erreur quadratique moyenne (RMSE) sur les points de contrôle.

In [ ]:
validation_errors = []

for frame_idx, all_points in IMAGE_POINTS_BY_FRAME_FALLBACK.items():
    if len(all_points) < 5: continue
    for i, (name, x_px, y_px) in enumerate(all_points):
        train_pts = [p for j, p in enumerate(all_points) if j != i]
        try:
            scaled_train = scale_image_points(train_pts)
            H_loo = compute_homography_for_frame(scaled_train)
            sx, sy = x_px * SCALE_X, y_px * SCALE_Y
            pt = np.array([[[float(sx), float(sy)]]], dtype=np.float32)
            pred = cv2.perspectiveTransform(pt, H_loo)[0, 0]
            expected_x, expected_y = FIELD_POINTS[name]
            error_m = math.sqrt((pred[0] - expected_x)**2 + (pred[1] - expected_y)**2)
            validation_errors.append({
                "frame": int(frame_idx), "point": name, "type": "leave_one_out",
                "expected_x_m": round(expected_x, 2), "expected_y_m": round(expected_y, 2),
                "pred_x_m": round(float(pred[0]), 2), "pred_y_m": round(float(pred[1]), 2), "error_m": round(error_m, 2)
            })
        except ValueError: pass

df_validation = pd.DataFrame(validation_errors)

loo_errors = df_validation.loc[
    df_validation["type"] == "leave_one_out",
    "error_m"
].dropna()

rmse_loo = (
    float(np.sqrt(np.mean(loo_errors ** 2)))
    if len(loo_errors)
    else None
)

mae_loo = (
    float(np.mean(np.abs(loo_errors)))
    if len(loo_errors)
    else None
)

df_validation.to_csv(
    OUTPUT_DIR / "validation_calibration.csv",
    index=False
)

print(f"RMSE Leave-One-Out : {rmse_loo:.2f} m")
print(f"MAE Leave-One-Out : {mae_loo:.2f} m")
display(df_validation)


## 11. Analyse Comparative (Benchmark)
Évaluation des performances (FPS/Précision) selon les modèles et trackers.

In [ ]:
benchmark_rows = []
bytetrack_configs = [
    {"detector": "YOLOv5u", "model": "yolov5su.pt", "tracker": "ByteTrack"},
    {"detector": "YOLOv8n", "model": "yolov8n.pt", "tracker": "ByteTrack"},
    {"detector": "YOLOv8m", "model": "yolov8m.pt", "tracker": "ByteTrack"},
]

# Optimisation : On limite aussi ByteTrack au nombre de frames défini pour DeepSORT
print(f"Lancement du benchmark (limite : {DEEPSORT_MAX_FRAMES} frames par configuration)...\n")

for cfg in bytetrack_configs:
    print(f"  Benchmarking {cfg['detector']} + {cfg['tracker']}...")
    model_bench = YOLO(cfg["model"])
    start_bench = time.time()
    frames, detections, tracks_with_id = 0, 0, 0

    # Utilisation de stream=True avec un compteur pour s'arrêter à DEEPSORT_MAX_FRAMES
    frames = 0
    detections = 0
    instances_with_id = 0
    unique_track_ids = set()

    for result in model_bench.track(
        source=VIDEO_PATH,
        tracker="bytetrack.yaml",
        persist=True,
        stream=True,
        save=False,
        classes=[0],
        verbose=False,
    ):
        frames += 1

        if result.boxes is not None:
            detections += len(result.boxes)

            if result.boxes.id is not None:
                ids = result.boxes.id.int().cpu().tolist()
                instances_with_id += len(ids)
                unique_track_ids.update(ids)

        if frames >= DEEPSORT_MAX_FRAMES:
            break

    elapsed_bench = time.time() - start_bench
    benchmark_rows.append({
        "configuration": f"{cfg['detector']} + {cfg['tracker']}",
        "detecteur": cfg["detector"],
        "tracker": cfg["tracker"],
        "frames": frames,
        "detections_personnes": detections,
        "instances_avec_id": instances_with_id,
        "identifiants_uniques": len(unique_track_ids),
        "temps_s": round(elapsed_bench, 2),
        "fps_traitement": round(frames / elapsed_bench, 2)
    })

def benchmark_deepsort(detector_name, model_path, max_frames=None):
    try:
        from deep_sort_realtime.deepsort_tracker import DeepSort
    except Exception as exc:
        return {
            "configuration": f"{detector_name} + DeepSORT",
            "detecteur": detector_name,
            "tracker": "DeepSORT",
            "statut": f"Indisponible: {exc}",
            "frames": 0,
            "detections_personnes": 0,
            "instances_avec_id": 0,
            "identifiants_uniques": 0,
            "temps_s": 0.0,
            "fps_traitement": 0
        }

    print(f"  Benchmarking {detector_name} + DeepSORT...")
    model_ds = YOLO(model_path)
    tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.3)
    cap = cv2.VideoCapture(VIDEO_PATH)
    frames, detections = 0, 0
    instances_with_id = 0
    unique_track_ids = set()
    start_bench = time.time()

    while True:
        ok, frame = cap.read()
        if not ok or (max_frames is not None and frames >= max_frames):
            break
        pred = model_ds.predict(frame, classes=[0], verbose=False)[0]
        ds_detections = []
        if pred.boxes is not None:
            boxes = pred.boxes.xyxy.cpu().numpy()
            confs = pred.boxes.conf.cpu().numpy()
            detections += len(boxes)
            for box, conf in zip(boxes, confs):
                ds_detections.append(([float(box[0]), float(box[1]), float(box[2]-box[0]), float(box[3]-box[1])], float(conf), "person"))

        tracks = tracker.update_tracks(ds_detections, frame=frame)

        for track in tracks:
            if track.is_confirmed() and track.time_since_update == 0:
                instances_with_id += 1
                unique_track_ids.add(track.track_id)

        frames += 1

    cap.release()
    elapsed_bench = time.time() - start_bench
    return {
        "configuration": f"{detector_name} + DeepSORT",
        "detecteur": detector_name,
        "tracker": "DeepSORT",
        "frames": frames,
        "detections_personnes": detections,
        "instances_avec_id": instances_with_id,
        "identifiants_uniques": len(unique_track_ids),
        "temps_s": round(elapsed_bench, 2),
        "fps_traitement": round(frames / elapsed_bench, 2)
    }

for cfg in [{"detector": "YOLOv5u", "model": "yolov5su.pt"}, {"detector": "YOLOv8n", "model": "yolov8n.pt"}, {"detector": "YOLOv8m", "model": "yolov8m.pt"}]:
    benchmark_rows.append(benchmark_deepsort(cfg["detector"], cfg["model"], DEEPSORT_MAX_FRAMES))

print("\nBenchmark terminé.")
df_benchmark = pd.DataFrame(benchmark_rows)
df_benchmark.to_csv(str(OUTPUT_DIR / "benchmark.csv"), index=False)
# Affichage du tableau de benchmark
display(df_benchmark)

## 12. Contrôle Qualité du Tracking
Analyse des ruptures de pistes et de la continuité des identifiants.

In [ ]:
gaps = df.sort_values(["tracker_id", "frame"]).copy()
gaps["frame_gap"] = gaps.groupby("tracker_id")["frame"].diff()
tracking_quality = gaps.groupby("tracker_id").agg(
    frames_observees=("frame", "count"),
    gaps_internes_piste=("frame_gap", lambda s: int((s > 1).sum())),
    gap_max_frames=("frame_gap", "max"),
    vitesse_max_km_h=("speed_km_h", "max"),
).fillna(0).round(2)
tracking_quality.to_csv(str(OUTPUT_DIR / "qualite_tracking.csv"))
# Affichage du tableau qualité du tracking
display(tracking_quality)

## 13. Visualisations de Synthèse
Cartes de chaleur, profils de vitesse et trajectoires tactiques.

In [ ]:
def draw_pitch_lines(ax):
    # Contour
    ax.plot([0, 105, 105, 0, 0], [0, 0, 68, 68, 0], color="white", lw=1.5, alpha=0.7)
    # Ligne mediane
    ax.plot([52.5, 52.5], [0, 68], color="white", lw=1.5, alpha=0.7)
    # Rond central
    theta = np.linspace(0, 2 * np.pi, 50)
    ax.plot(52.5 + 9.15 * np.cos(theta), 34 + 9.15 * np.sin(theta), color="white", lw=1.5, alpha=0.7)
    # Surfaces de reparation
    ax.plot([0, 16.5, 16.5, 0], [13.84, 13.84, 54.16, 54.16], color="white", lw=1.5, alpha=0.7)
    ax.plot([105, 88.5, 88.5, 105], [13.84, 13.84, 54.16, 54.16], color="white", lw=1.5, alpha=0.7)

df_plot = df.dropna(subset=["x_m", "y_m", "speed_km_h"]).copy()
df_plot = df_plot[df_plot["speed_km_h"] >= 0]
main_tracks = (
    stats_joueurs
    .reset_index()
    .sort_values("frames_observees", ascending=False)
    .head(3)["tracker_id"]
    .astype(int)
    .tolist()
)

# --- Figure 1: Heatmap + Vitesse + Trajectoire ---
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Heatmap Équipe A
team_a = df_plot[df_plot["team"] == 0]
if len(team_a) > 0:
    axes[0, 0].set_facecolor("#2e7d32")
    sns.kdeplot(data=team_a, x="x_m", y="y_m", fill=True, cmap="YlOrRd", thresh=0.05, ax=axes[0, 0], alpha=0.85)
    draw_pitch_lines(axes[0, 0])
    axes[0, 0].set_xlim(0, PITCH_LENGTH_M)
    axes[0, 0].set_ylim(0, PITCH_WIDTH_M)
    axes[0, 0].set_title(f"Carte de chaleur (Heatmap) - {TEAM_DISPLAY_LABELS[0]}", fontsize=12, fontweight="bold")
    axes[0, 0].set_xlabel("Longueur du terrain (m)", fontsize=10)
    axes[0, 0].set_ylabel("Largeur du terrain (m)", fontsize=10)

# Heatmap Équipe B
team_b = df_plot[df_plot["team"] == 1]
if len(team_b) > 0:
    axes[0, 1].set_facecolor("#2e7d32")
    sns.kdeplot(data=team_b, x="x_m", y="y_m", fill=True, cmap="YlOrRd", thresh=0.05, ax=axes[0, 1], alpha=0.85)
    draw_pitch_lines(axes[0, 1])
    axes[0, 1].set_xlim(0, PITCH_LENGTH_M)
    axes[0, 1].set_ylim(0, PITCH_WIDTH_M)
    axes[0, 1].set_title(f"Carte de chaleur (Heatmap) - {TEAM_DISPLAY_LABELS[1]}", fontsize=12, fontweight="bold")
    axes[0, 1].set_xlabel("Longueur du terrain (m)", fontsize=10)
    axes[0, 1].set_ylabel("Largeur du terrain (m)", fontsize=10)

# Vitesse top 3 joueurs
for tid in main_tracks:
    track_data = df_plot[df_plot["tracker_id"] == tid]
    axes[1, 0].plot(track_data["time_s"], track_data["speed_km_h"], linewidth=1.5, label=f"Joueur ID {tid}")
axes[1, 0].set_title(
    "Profil de vitesse des 3 pistes les plus longtemps observées",
    fontsize=12,
    fontweight="bold"
)
axes[1, 0].set_xlabel("Temps écoulé (s)", fontsize=10)
axes[1, 0].set_ylabel("Vitesse instantanée (km/h)", fontsize=10)
axes[1, 0].legend(loc="upper right", frameon=True)
axes[1, 0].grid(True, alpha=0.3)

# Trajectoire du joueur principal (Joueur 499)
main_tid = main_tracks[0]
track_main = df_plot[df_plot["tracker_id"] == main_tid]
axes[1, 1].set_facecolor("#2e7d32")
draw_pitch_lines(axes[1, 1])
axes[1, 1].plot(track_main["x_m"], track_main["y_m"], linewidth=1.5, color="#1e88e5", label="Trajectoire")
axes[1, 1].scatter(track_main["x_m"].iloc[0], track_main["y_m"].iloc[0], c="#4caf50", s=120, edgecolors="black", label="Position initiale (Départ)", zorder=5)
axes[1, 1].scatter(track_main["x_m"].iloc[-1], track_main["y_m"].iloc[-1], c="#f44336", s=120, edgecolors="black", label="Position finale (Arrivée)", zorder=5)
axes[1, 1].set_xlim(0, PITCH_LENGTH_M)
axes[1, 1].set_ylim(0, PITCH_WIDTH_M)
axes[1, 1].set_title(
    f"Trajectoire de la piste de suivi ID {main_tid}",
    fontsize=12,
    fontweight="bold"
)
axes[1, 1].set_xlabel("Longueur du terrain (m)", fontsize=10)
axes[1, 1].set_ylabel("Largeur du terrain (m)", fontsize=10)
axes[1, 1].legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig(str(viz_dir / "analyse_complete.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(viz_dir / "analyse_complete.svg"), dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 2: Distribution des vitesses ---
fig, ax = plt.subplots(figsize=(11, 5))
for team_id, label, color in [(0, TEAM_DISPLAY_LABELS[0], TEAM_COLORS_HEX[0]), (1, TEAM_DISPLAY_LABELS[1], TEAM_COLORS_HEX[1])]:
    team_data = df_plot[(df_plot["team"] == team_id) & (df_plot["speed_km_h"] > 0.5)]["speed_km_h"]
    if len(team_data) > 0:
        sns.kdeplot(team_data, label=label, color=color, fill=True, alpha=0.3, ax=ax, linewidth=2)
ax.set_title("Distribution de la densité de probabilité des vitesses par équipe", fontsize=14, fontweight='bold')
ax.set_xlabel("Vitesse des joueurs (km/h)", fontsize=12)
ax.set_ylabel("Densité de probabilité (KDE)", fontsize=12)
ax.set_xlim(0, 35)
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
plt.savefig(str(viz_dir / "distribution_vitesses.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(viz_dir / "distribution_vitesses.svg"), dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 3: Distance totale par joueur ---
top_dist = stats_joueurs.head(20).reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
colors_bars = [
    TEAM_COLORS_HEX.get(int(team_id), TEAM_COLORS_HEX[-1])
    for team_id in top_dist["team"]
]
bars = ax.bar(range(len(top_dist)), top_dist["distance_totale_m"], color=colors_bars, alpha=0.8, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(top_dist)))
ax.set_xticklabels([f"J{int(tid)}" for tid in top_dist["tracker_id"]], rotation=45, ha="right")
ax.set_title("Distance cumulée — 20 pistes les plus longtemps observées")
ax.set_xlabel("Identifiant de suivi (Tracker ID)")
ax.set_ylabel("Distance totale parcourue (m)", fontsize=12)

from matplotlib.patches import Patch
legend_elements = [
    Patch(
        facecolor=TEAM_COLORS_HEX[0],
        edgecolor="black",
        label=TEAM_DISPLAY_LABELS[0]
    ),
    Patch(
        facecolor=TEAM_COLORS_HEX[1],
        edgecolor="black",
        label=TEAM_DISPLAY_LABELS[1]
    ),
    Patch(
        facecolor=TEAM_COLORS_HEX[-1],
        edgecolor="black",
        label=TEAM_DISPLAY_LABELS[-1]
    ),
]
ax.legend(handles=legend_elements, frameon=True)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(str(viz_dir / "distances_joueurs.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(viz_dir / "distances_joueurs.svg"), dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 4: Benchmark comparatif FPS ---
fig, ax = plt.subplots(figsize=(11, 5))
colors_bench = ["#2196F3" if "ByteTrack" in c else "#FF5722" for c in df_benchmark["configuration"]]
y_positions = range(len(df_benchmark))
ax.barh(y_positions, df_benchmark["fps_traitement"], color=colors_bench, alpha=0.8, edgecolor="black", height=0.6)
ax.set_yticks(y_positions)
ax.set_yticklabels(df_benchmark["configuration"])
ax.axvline(x=VIDEO_FPS, color="#2e7d32", linestyle="--", linewidth=2.5, label=f"Seuil du temps réel vidéo ({VIDEO_FPS:.0f} images/seconde)")
ax.set_xlabel("Vitesse d'exécution / Débit de traitement (images par seconde - FPS)", fontsize=12)
ax.set_title("Comparatif de l'efficacité computationnelle par configuration", fontsize=14, fontweight='bold')
ax.legend(loc="lower right", frameon=True)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(str(viz_dir / "benchmark_fps.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(viz_dir / "benchmark_fps.svg"), dpi=150, bbox_inches="tight")
plt.show()


### Focus : Vue Split-Screen (Caméra vs Terrain 2D)
Projection synchronisée pour l'analyse tactique.

In [ ]:
# Cellule de visualisation de tracking avec projection 2D
def generate_tracking_visualization_frames(frame_indices, df_coords, video_path, output_viz_dir, h_by_frame):
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from pathlib import Path

    output_viz_dir = Path(output_viz_dir)
    output_viz_dir.mkdir(exist_ok=True, parents=True)

    # Dictionnaire de correspondance de couleurs pour l'affichage (RGB)
    # Équipe A -> bleu, Équipe B -> rouge, non classé ou autre -> jaune
    team_colors = {
        0: (30, 136, 229),   # Bleu (#1e88e5)
        1: (229, 57, 53),    # Rouge (#e53935)
        -1: (253, 216, 53)   # Jaune (#fdd835)
    }

    cap = cv2.VideoCapture(str(video_path))
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            print(f"Impossible de lire la trame {frame_idx}")
            continue

        # 1. Selectionner les donnees de tracking pour cette trame
        frame_data = df_coords[df_coords["frame"] == frame_idx].dropna(subset=["x_px", "y_px"])
        if len(frame_data) == 0:
            print(f"Aucune donnee de tracking pour la trame {frame_idx}")
            continue

        # Copie de l'image de la camera pour dessiner
        img_drawn = frame.copy()

        # 2. Creation de la figure Matplotlib split-screen (1 ligne, 2 colonnes)
        fig, (ax_cam, ax_pitch) = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={'width_ratios': [1.2, 1.0]})

        # --- PANEL GAUCHE : VUE CAMERA ORIGINAL AVEC ANNOTATIONS ---
        img_rgb = cv2.cvtColor(img_drawn, cv2.COLOR_BGR2RGB)

        # Dessiner les boites englobantes et les tracker IDs sur l'image camera
        for _, row in frame_data.iterrows():
            t_id = int(row["tracker_id"])
            team = int(row.get("team", -1))
            color_rgb = team_colors.get(team, team_colors[-1])

            x_min, y_min = int(row.get("x_min", 0)), int(row.get("y_min", 0))
            x_max, y_max = int(row.get("x_max", 0)), int(row.get("y_max", 0))

            if x_max > x_min and y_max > y_min:
                # Dessiner le rectangle
                cv2.rectangle(img_rgb, (x_min, y_min), (x_max, y_max), color_rgb, 3)
                # Dessiner le label d'identifiant
                label = f"J{t_id}"
                (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                cv2.rectangle(img_rgb, (x_min, y_min - text_h - 10), (x_min + text_w + 10, y_min), color_rgb, -1)
                cv2.putText(img_rgb, label, (x_min + 5, y_min - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        ax_cam.imshow(img_rgb)
        ax_cam.set_title(f"Vue Caméra Originale - Trame {frame_idx}\n(Détection YOLOv8m + Suivi ByteTrack)", fontsize=14, fontweight='bold')
        ax_cam.axis("off")

        # --- PANEL DROIT : REPRESENTATION TACTIQUE 2D ---
        ax_pitch.set_facecolor("#2e7d32") # Pelouse verte
        draw_pitch_lines(ax_pitch)

        # Tracer les joueurs projetes sur le terrain 2D
        for _, row in frame_data.iterrows():
            x_m = row.get("x_m")
            y_m = row.get("y_m")
            if pd.isna(x_m) or pd.isna(y_m):
                continue

            t_id = int(row["tracker_id"])
            team = int(row.get("team", -1))
            color_hex = '#1e88e5' if team == 0 else ('#e53935' if team == 1 else '#fdd835')

            # Dessiner le point du joueur
            ax_pitch.scatter(x_m, y_m, c=color_hex, s=150, edgecolors="white", linewidth=1.5, zorder=5)
            # Ajouter le texte du tracker_id a cote du point
            ax_pitch.text(x_m + 1.2, y_m + 0.8, f"J{t_id}", color="white", fontsize=10, fontweight='bold',
                          bbox=dict(facecolor=color_hex, alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'), zorder=6)

        ax_pitch.set_xlim(0, PITCH_LENGTH_M)
        ax_pitch.set_ylim(0, PITCH_WIDTH_M)
        # Inverser l'axe Y pour correspondre au sens de la camera
        ax_pitch.set_ylim(PITCH_WIDTH_M, 0)
        ax_pitch.set_title("Projection Tactique 2D (Plan du Terrain)\n(Coordonnées réelles projetées en mètres)", fontsize=14, fontweight='bold')
        ax_pitch.set_xlabel("Longueur du terrain (m)", fontsize=11)
        ax_pitch.set_ylabel("Largeur du terrain (m)", fontsize=11)

        # Legende
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=TEAM_COLORS_HEX[0], edgecolor="white", label=TEAM_DISPLAY_LABELS[0]),
            Patch(facecolor=TEAM_COLORS_HEX[1], edgecolor="white", label=TEAM_DISPLAY_LABELS[1]),
            Patch(facecolor=TEAM_COLORS_HEX[-1], edgecolor="white", label=TEAM_DISPLAY_LABELS[-1])
        ]
        ax_pitch.legend(handles=legend_elements, loc="upper right", frameon=True, facecolor='#ffffff', edgecolor='#cccccc', fontsize=10)

        plt.tight_layout()
        viz_path = output_viz_dir / f"tracking_viz_frame_{frame_idx:06d}.png"
        plt.savefig(str(viz_path), dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"Visualisation enregistrée pour la trame {frame_idx} : {viz_path.name}")

    cap.release()

# Appel de la fonction pour generer les images aux trames cles
viz_keyframes = [150, 300, 600, 750, 900]
print("\nGénération des visualisations de tracking split-screen (Caméra vs Terrain 2D)...")
generate_tracking_visualization_frames(
    frame_indices=viz_keyframes,
    df_coords=df,
    video_path=VIDEO_PATH,
    output_viz_dir=viz_dir,
    h_by_frame=H_BY_FRAME
)

## 14. Diagnostic de Santé des Données
Vérification automatique de la cohérence physique des résultats.

In [ ]:
# Evaluation automatique de la qualite des donnees generees
print("\n" + "=" * 60)
print("DIAGNOSTIC DE SANITE DES RESULTATS")
print("=" * 60)
diag_ok = True

if rmse_loo is not None:
    if rmse_loo < 5.0: print(f"RMSE calibration: {rmse_loo:.2f} m (conforme)")
    else:
        print(f"RMSE calibration trop eleve")
        diag_ok = False

speeds_valid = df["speed_km_h"].dropna()
pct_high = (speeds_valid > 30).mean() * 100
if pct_high > 5:
    print(f"Anomalie vitesse detectee: {pct_high:.1f}% > 30 km/h")
    diag_ok = False

pct_positions_valides = (
    df["position_dans_terrain"].mean() * 100
)

pct_vitesses_valides = (
    df["speed_km_h"].notna().mean() * 100
)

pct_pistes_affectees_cluster = (
    df.groupby("tracker_id")["team"]
    .first()
    .ge(0)
    .mean()
    * 100
)

candidate_speeds = df["speed_km_h_raw"].notna()
pct_vitesses_rejetees = (
    (candidate_speeds & df["speed_km_h"].isna()).sum()
    / candidate_speeds.sum()
    * 100
    if candidate_speeds.sum() > 0
    else 0.0
)

print(f"Positions projetées valides : {pct_positions_valides:.1f}%")
print(f"Vitesses valides : {pct_vitesses_valides:.1f}%")
print(f"Pistes affectées à l’un des deux clusters : {pct_pistes_affectees_cluster:.1f}%")
print(f"Vitesses rejetées par les filtres : {pct_vitesses_rejetees:.1f}%")

## 15. Export et Rapport Final
Génération du fichier JSON de synthèse.

In [ ]:
summary = {
    "version": "v1.0-final",
    "video_url": VIDEO_URL,
    "video_frames": FRAME_COUNT,
    "video_fps": round(VIDEO_FPS, 2),
    "resolution": f"{WIDTH}x{HEIGHT}",
    "reference_resolution": f"{REFERENCE_WIDTH}x{REFERENCE_HEIGHT}",
    "auto_scaling": SCALE_X != 1.0 or SCALE_Y != 1.0,
    "keyframes_calibrees": len(H_BY_FRAME),
    "keyframes_prevues": len(valid_keyframes),
    "nb_tracks": int(df["tracker_id"].nunique()),
    "nb_equipes": int(df[df["team"] >= 0]["team"].nunique()),
    "nb_detections_projetees": int(len(df)),
    "filtre_kalman_applique": True,
    "filtre_acceleration_applique": True,
    "clustering_equipes": True,
    "meilleure_piste_frames": int(stats_joueurs.iloc[0]["frames_observees"]) if len(stats_joueurs) else 0,
    "meilleure_piste_distance_m": float(stats_joueurs.iloc[0]["distance_totale_m"]) if len(stats_joueurs) else 0,
    "rmse_loo_m": round(rmse_loo, 2) if rmse_loo is not None else None,
    "mae_loo_m": round(mae_loo, 2) if mae_loo is not None else None,
    "max_speed_threshold_kmh": MAX_SPEED_M_S * 3.6,
    "max_accel_threshold_ms2": MAX_ACCEL_M_S2,
    "diagnostic_ok": diag_ok,
    "pct_positions_valides": round(pct_positions_valides, 2),
    "pct_vitesses_valides": round(pct_vitesses_valides, 2),
    "pct_pistes_affectees_cluster": round(pct_pistes_affectees_cluster, 2),
    "pct_vitesses_rejetees": round(pct_vitesses_rejetees, 2),
}

(OUTPUT_DIR / "synthese_resultats.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("\n=== SYNTHESE v1.0-final SECURISEE ===")
for k, v in summary.items(): print(f"  {k}: {v}")